# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates exploratory data loading and processing for the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore high-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)
print("Dataset loaded.")

# Display high-level metadata
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
List available record sets, fields, and their `@id` references.
This will help in selecting the appropriate record set and fields for extraction.

In [ ]:
# Explore available record sets and inspect fields/columns
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and len(rs.fields) > 0:
        print("  Fields (with @id):")
        for fld in rs.fields:
            print(f"    - {fld.name} (id: {fld.id}, datatype: {fld.data_type})")
    elif hasattr(rs, 'columns') and len(rs.columns) > 0:
        print("  Columns (with @id):")
        for col in rs.columns:
            print(f"    - {col.name} (id: {col.id}, datatype: {col.data_type})")
    print()

## 3. Data Extraction
Load data from a selected record set into a pandas DataFrame for analysis. Use the `@id` for the record set you wish to extract.

In [ ]:
# Pick the primary record set (by @id, adjust as per overview result)
# For example, suppose the main record set's id is 'cr:RecordSet1'
record_set_id = record_sets[0].id if record_sets else None
assert record_set_id is not None, "No record set found!"

# Extract all records from the selected record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from record set {record_set_id}.")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let’s apply some common EDA steps, such as filtering, normalization, and grouping. Choose field `@id`s from the earlier listing for your analysis.

In [ ]:
# Replace these with actual field/column @id strings (see output from Step 2, typically like 'cr:Field_Abundance')
# For illustration, pick two typical columns: a numeric field and a grouping/categorical field
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    if group_field_id is None and pd.api.types.is_string_dtype(df[col]):
        group_field_id = col

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group/categorical field: {group_field_id}")

# Filter records with high values in the numeric field
if numeric_field_id is not None:
    threshold = df[numeric_field_id].quantile(0.9)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} above 90th percentile (>{threshold}): {len(filtered_df)} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by category and aggregate
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped means by {group_field_id} (top values shown):")
        print(grouped_df.head())
else:
    print("No numeric field found in the record set.")

## 5. Visualization
Let’s plot the distribution of the numeric field and the group-wise aggregation, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# If group-by available, plot bar graph
if group_field_id is not None and numeric_field_id is not None:
    grouped_df = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    grouped_df_sorted = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df_sorted)
    plt.title(f"Top 10 mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using `mlcroissant`, explored its record sets and fields, performed initial filtering and normalization, grouped and visualized the data. This analysis can be extended for deeper investigation, such as studying protein expression variation, identifying outlier proteins, or benchmarking across experimental batches.